# Cirq interoperability with native registers

Adapted from Qualtran v0.7.0, Apache-2.0.

In [ ]:
import cirq
import numpy as np
from cirq.contrib.svg import SVGCircuit
from IPython.display import display
from flagquantum.circuit import Circuit
from flagquantum.core.ir import CircuitIR, Instruction
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.core.ir.program.registers import Register, RegisterSide
from flagquantum.core.ir.program.register_types import opaque_register_type
from flagquantum.ecosystem.cirq import export_cirq, import_cirq
from flagquantum.ecosystem.cirq._register_export import export_register_cirq

## Importing Cirq circuits

The optional adapter converts Cirq circuits into native IR. Its report describes operations and metadata outside the static contract.

### A five-qubit Fourier transform

Inspect a Cirq Fourier gate as one through register with five binary elements, then compare its matrix with the native Fourier circuit.

In [ ]:
import torch

from flagquantum.algorithms.primitives.qft import qft as native_qft
from flagquantum.ecosystem.cirq._gate_metadata import cirq_gate_signature, cirq_gate_tensor
from flagquantum.ecosystem.cirq._register_gate import RegisterCirqGate
from flagquantum.simulation.unitary import get_unitary

fourier = cirq.QuantumFourierTransformGate(num_qubits=5)
fourier_signature = cirq_gate_signature(fourier)
display(fourier_signature)

In [ ]:
fourier_reference = cirq_gate_tensor(fourier).reshape(32, 32)
fourier_circuit = native_qft(5)
fourier_matrix = get_unitary(fourier_circuit, dtype=torch.complex128, device="cpu").numpy()
np.testing.assert_allclose(fourier_matrix, fourier_reference, atol=2e-12, rtol=0)

fourier_wrapper = RegisterCirqGate(fourier_circuit.to_ir(), fourier_signature)
np.testing.assert_allclose(cirq.unitary(fourier_wrapper), fourier_reference, atol=2e-12, rtol=0)
display(SVGCircuit(cirq.Circuit(fourier_wrapper.on(*cirq.LineQubit.range(5)))))

### Convert the seeded example

The original random circuit contains an iSWAP. Use its exact decomposition before importing, and compare both matrices.

In [ ]:
qubits = cirq.LineQubit.range(4)
original = cirq.testing.random_circuit(qubits, n_moments=5, op_density=1.0, random_state=52)
lowered = cirq.Circuit()
for op in original.all_operations():
    if op.gate == cirq.ISWAP:
        a, b = op.qubits
        lowered.append([cirq.S(a), cirq.S(b), cirq.CZ(a, b), cirq.SWAP(a, b)])
    else:
        lowered.append(op)
imported = import_cirq(lowered, allow_lossy=True)
assert {issue.code for issue in imported.report.issues} <= {"moment_structure_flattened"}
display(SVGCircuit(original))

### Check the full unitary

In [ ]:
exported = export_cirq(imported.ir)
np.testing.assert_allclose(original.unitary(qubit_order=qubits), exported.circuit.unitary(qubit_order=qubits), atol=2e-12, rtol=0)

### Export the imported IR on explicit qubits

In [ ]:
word = Signature((Register("qubits", opaque_register_type(1), shape=(4,)),))
roundtrip, ports = export_register_cirq(
    imported.ir, word, {"qubits": (0, 1, 2, 3)}, input_quregs={"qubits": [[q] for q in qubits]}
)
display(SVGCircuit(roundtrip.circuit))

In [ ]:
display(original == roundtrip.circuit)

In [ ]:
np.testing.assert_allclose(original.unitary(qubit_order=qubits), roundtrip.circuit.unitary(qubit_order=qubits), atol=2e-12, rtol=0)
display(imported.report.to_dict())

## Exporting native register operations

### Bind an exchange to named wires

In [ ]:
from examples.algorithms.binary_field_arithmetic import FieldOperation

swap_body = (Instruction("swap", (0, 1)),)
swap = FieldOperation(2, swap_body)
pair = Signature.build(x=1, y=1)
binding = {"x": (0,), "y": (1,)}
program = Circuit(2)
program.append(swap)

In [ ]:
external, output = export_register_cirq(
    program, pair, binding, input_quregs={"x": cirq.NamedQubit("q1"), "y": cirq.NamedQubit("q2")}
)
display(SVGCircuit(external.circuit))

### Bind a composite circuit

In [ ]:
named, named_outputs = export_register_cirq(program, pair, binding)
display(named_outputs)

In [ ]:
program.append(swap)
twice, _ = export_register_cirq(program, pair, binding)
np.testing.assert_allclose(twice.circuit.unitary(), np.eye(4), atol=2e-12, rtol=0)
display(SVGCircuit(twice.circuit))

### Inspect a word exchange

The native compilation emits the component gates.

In [ ]:
from flagquantum.compiler.registers.primitives.swap import swap_register_ir

In [ ]:
left, right = (0, 1, 2), (3, 4, 5)
signature = Signature.build(x=3, y=3)
wide = Circuit(6)
wide.append(FieldOperation(6, swap_register_ir(left, right)))

Place the two words on distant external line ranges.

In [ ]:
placed, wide_ports = export_register_cirq(
    wide, signature, {"x": left, "y": right},
    input_quregs={"x": cirq.LineQubit.range(3), "y": cirq.LineQubit.range(100, 103)},
)

Inspect the emitted leaf operations and their operand order.

In [ ]:
ops = list(placed.circuit.all_operations())
expected_pairs = list(zip(wide_ports["x"], wide_ports["y"]))
assert [op.qubits for op in ops] == expected_pairs
display(ops)

In [ ]:
display(SVGCircuit(placed.circuit))

### Left, right and through ports

In [ ]:
from flagquantum.compiler.registers.control.multi_control import multi_and_ir

left_right = Signature((
    Register("ctrl", opaque_register_type(1)),
    Register("result", opaque_register_type(1), side=RegisterSide.RIGHT),
))
fresh = CircuitIR(2, (Instruction("cx", (0, 1)),))

Supply only the input register arrays. A right-only port is placed on fresh external qubits.

In [ ]:
control = cirq.NamedQubit("input")
fresh_output, fresh_ports = export_register_cirq(fresh, left_right, {"ctrl": (0,), "result": (1,)}, input_quregs={"ctrl": control})
assert fresh_ports["ctrl"][0] == control
assert fresh_ports["result"][0] != control

In [ ]:
display(SVGCircuit(fresh_output.circuit))

### A four-control conjunction

The four-control MultiAnd retains the two intermediate conjunctions in the right-only junk register. The last conjunction appears in the right-only target. The named register export preserves the inverse phase gates in the compute sequence.

In [ ]:
and_signature = Signature((
    Register("ctrl", opaque_register_type(1), shape=(4,)),
    Register("junk", opaque_register_type(1), shape=(2,), side=RegisterSide.RIGHT),
    Register("target", opaque_register_type(1), side=RegisterSide.RIGHT),
))
ladder = CircuitIR(7, multi_and_ir((0, 1, 2, 3), (1, 1, 1, 1), (4, 5, 6)))
and_ports = {"ctrl": (0, 1, 2, 3), "junk": (4, 5), "target": (6,)}
and_export, and_outputs = export_register_cirq(ladder, and_signature, and_ports)
assert and_export.report.lossless
assert len(tuple(and_export.circuit.all_operations())) == 39
assert tuple(and_outputs) == ("ctrl", "junk", "target")
assert and_outputs["junk"].shape == (2, 1)
assert and_outputs["target"].shape == (1,)
display(and_outputs)
display(SVGCircuit(and_export.circuit))

## Modular exponentiation roundtrip

Inspect the original large example's signature and call graph. Compile a small valid instance, export its feedback, and import the wrapped gate through its owned native definition. The typed roundtrip preserves each measurement record and conditional operand.

In [ ]:
from flagquantum.algorithms.primitives.modular_exponentiation import ModularExponentiation
from flagquantum.dynamic import DynamicCircuit

large = ModularExponentiation(8, 13 * 17, 32, 32)
display(large.source_signature)
display(large.source_call_graph().direct_calls)
assert large.classical_values(13) == (13, pow(8, 13, 221))

In [ ]:
modexp = ModularExponentiation(2, 3, 1, 2)
work, slots = modexp.workspace_requirements()
modexp_width = 3 + work
native = DynamicCircuit(modexp_width)
native.append(modexp.make_on(
    exponent=(0,), target=(1, 2), workspace=tuple(range(3, modexp_width)),
    classical_bits=tuple(range(slots)),
))
body = native.to_ir()
modexp_ports = {"exponent": (0,), "x": (1, 2)}
modexp_export, modexp_outputs = export_register_cirq(
    body, modexp.source_signature, modexp_ports, feedback_key_prefix="modexp",
)
assert modexp_export.report.lossless
assert len(cirq.measurement_key_names(modexp_export.circuit)) == 48
ordered = (*modexp_outputs["exponent"], *modexp_outputs["x"], *cirq.LineQubit.range(3, modexp_width))
trajectory = cirq.Simulator(dtype=np.complex128, seed=34).simulate(
    modexp_export.circuit, qubit_order=ordered, initial_state=1 << (modexp_width - 1),
)
expected = np.zeros(1 << modexp_width, dtype=np.complex128)
expected[(1 << (modexp_width - 1)) | (2 << (modexp_width - 3))] = 1
np.testing.assert_allclose(trajectory.final_state_vector, expected, atol=2e-12, rtol=0)

In [ ]:
from flagquantum.compiler.registers.component_gate_function import bit_gate_function
from flagquantum.compiler.registers.register_binding import bind_register_program
from flagquantum.core.ir.program.register_types import quantum_bit_type
from flagquantum.ecosystem.cirq._register_import import import_cirq_register_optree

physical = Signature((Register("w", quantum_bit_type(), shape=(modexp_width,)),))
wires = tuple(range(modexp_width))
qids = tuple(cirq.LineQubit.range(modexp_width))
placement = {"w": np.asarray(qids, dtype=object).reshape(modexp_width, 1)}
definition = bit_gate_function("modexp", tuple(physical), body)
wrapper = RegisterCirqGate(body, physical, feedback_key_prefix="modexp")
restored = import_cirq_register_optree(
    wrapper.on(*qids), signature=physical, in_quregs=placement, out_quregs=placement,
    gate_definitions={wrapper: definition},
)
bound = bind_register_program(restored, tuple((wire,) for wire in wires))
reexported, _ = export_register_cirq(
    bound.circuit, physical, {"w": wires}, input_quregs=placement,
    feedback_key_prefix="modexp",
)
assert reexported.report.lossless
assert tuple(reexported.circuit.all_operations()) == tuple(cirq.decompose_once(wrapper.on(*qids)))
display({"operations": len(bound.circuit.instructions), "measurements": 48, "work_wires": work})